<a href="https://colab.research.google.com/github/aiza-snflwer/cardiometabolic-risk-prediction/blob/main/HTN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Hypertension Risk Prediction using Logistic Regression
# Includes: data preprocessing, imputation, baseline model, SMOTE for class imbalance, and evaluation

In [ ]:
import pandas as pd
import numpy as np
import json

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from imblearn.over_sampling import SMOTE
from google.colab import files

In [ ]:
# Upload dataset
uploaded = files.upload()
file_name = list(uploaded.keys())[0]

# Load dataset
data = pd.read_excel(file_name)

# Basic overview
print("Dataset shape:", data.shape)
data.head()

Saving hypertension_dataset_clean.xlsx to hypertension_dataset_clean.xlsx
Dataset shape: (174982, 8)


,Age,BMI,Systolic_BP,Diastolic_BP,Physical_Activity_Level,Family_History,Stress_Level,Hypertension
0,58,29.5,160,79,1,1,9,1
1,34,36.2,120,84,3,1,6,1
2,73,18.2,156,60,3,1,5,0
3,60,20.3,122,94,2,1,6,1
4,73,21.8,91,97,2,1,6,1


In [ ]:
# Features used for hypertension prediction
X = data[['Age', 'BMI', 'Systolic_BP', 'Diastolic_BP',
          'Physical_Activity_Level', 'Family_History', 'Stress_Level']]

# Target variable (0 = no hypertension, 1 = hypertension)
y = data['Hypertension']

print("Class distribution:")
print(y.value_counts())

Class distribution:
Hypertension
1    125781
0     49201
Name: count, dtype: int64


In [ ]:
# Split dataset into training and testing sets (80/20 split)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
# Handle missing values using mean imputation
imputer = SimpleImputer(strategy="mean")

X_train = imputer.fit_transform(X_train)
X_test = imputer.transform(X_test)

In [ ]:
# Train baseline Logistic Regression model
model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train, y_train)

# Predictions
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)

# Convert probabilities into risk percentage
risk_percentage = y_prob[:, 1] * 100

In [ ]:
print("=== BASELINE MODEL ===")
print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))

=== BASELINE MODEL ===
Accuracy: 0.7174329228219561
              precision    recall  f1-score   support

           0       0.00      0.00      0.00      9889
           1       0.72      1.00      0.84     25108

    accuracy                           0.72     34997
   macro avg       0.36      0.50      0.42     34997
weighted avg       0.51      0.72      0.60     34997

[[    0  9889]
 [    0 25108]]


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [ ]:
# Apply SMOTE to balance training dataset
smote = SMOTE(random_state=42)

X_resampled, y_resampled = smote.fit_resample(X_train, y_train)

print("Class distribution after SMOTE:")
print(y_resampled.value_counts())

Class distribution after SMOTE:
Hypertension
0    100673
1    100673
Name: count, dtype: int64


In [ ]:
# Train model on balanced dataset
model_resampled = LogisticRegression(max_iter=1000, random_state=42)
model_resampled.fit(X_resampled, y_resampled)

# Predictions
y_pred_resampled = model_resampled.predict(X_test)

In [ ]:
print("=== SMOTE MODEL ===")
print(classification_report(y_test, y_pred_resampled))
print(confusion_matrix(y_test, y_pred_resampled))

=== SMOTE MODEL ===
              precision    recall  f1-score   support

           0       0.28      0.51      0.36      9889
           1       0.72      0.49      0.58     25108

    accuracy                           0.50     34997
   macro avg       0.50      0.50      0.47     34997
weighted avg       0.60      0.50      0.52     34997

[[ 5034  4855]
 [12733 12375]]


In [ ]:
all_patients_json = []

for i in range(len(X_test)):

    patient_data = X_test[i]
    risk = risk_percentage[i]

    # Risk classification
    if risk > 50:
        level = "High"
    elif risk > 20:
        level = "Moderate"
    else:
        level = "Low"

    # Contributing factors
    factors = []

    # Blood pressure
    if patient_data[2] > 130 or patient_data[3] > 80:
        factors.append({
            "factor": "High Blood Pressure",
            "impact": "High"
        })

    # BMI
    if patient_data[1] > 25:
        factors.append({
            "factor": "High BMI",
            "impact": "Moderate"
        })

    # Age
    if patient_data[0] > 40:
        factors.append({
            "factor": "Age",
            "impact": "Moderate"
        })

    # Physical activity
    if patient_data[4] == 1:
        factors.append({
            "factor": "Low Physical Activity",
            "impact": "Moderate"
        })

    # Family history
    if patient_data[5] == 1:
        factors.append({
            "factor": "Family History",
            "impact": "Moderate"
        })

    # Stress
    if patient_data[6] > 7:
        factors.append({
            "factor": "High Stress",
            "impact": "Moderate"
        })

    all_patients_json.append({
        "patient_id": i,
        "risk_percentage": round(float(risk), 2),
        "risk_level": level,
        "contributing_factors": factors
    })

# Save JSON
with open("HTN_PREDICTIONS.json", "w") as f:
    json.dump(all_patients_json, f, indent=2)

print("JSON saved successfully.")

JSON saved successfully.
